# TP=1 vs TP=2 benchmark and acceptance verdict

**EXPERIMENTAL — KAGGLE DUAL-T4 VALIDATION PENDING.** Not executed by the author; outputs are produced only when *you* run it.

Accelerator: **GPU T4 x2**, Internet on only if the model must be downloaded. Run notebooks in order 01 → 06; each stage writes JSON to `/kaggle/working/stokastisk-acceptance/`.

Same model, prompts and sampling for both runs (one process per TP). Reports measured decode tokens/s, prefill tokens/s, latency and GPU memory, and re-checks the `TRITON_ATTN` engine config; **no speed-up is assumed** (the T4s have no P2P). Ends with the overall acceptance verdict.

In [ ]:
import glob, os, pathlib
# Locate the release bundle (the Kaggle dataset holding manifests/SHA256SUMS). Override with STOKASTISK_BUNDLE.
hits = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
BUNDLE = pathlib.Path(os.environ.get("STOKASTISK_BUNDLE") or (hits[0] if hits else "")) 
assert (BUNDLE / "manifests" / "SHA256SUMS").is_file(), "attach the stokastisk-0.3.0 release dataset to this notebook"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("bundle:", BUNDLE)

In [ ]:
import subprocess, sys, os
env = dict(os.environ, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""))
for tp in (1, 2):
    rc = subprocess.run([sys.executable, str(SCRIPTS / "06_tp_benchmark.py"), str(tp), "Qwen/Qwen2.5-1.5B-Instruct"], env=env).returncode
    print("TP", tp, "rc", rc)

In [ ]:
!cd {SCRIPTS} && python 07_summary.py